In [13]:
import pandas as pd 
import math 
PATH_CONCRETE = r"Z:\Tier_0\06_Carbon_Rates\Process-based_Carbon_Benchmarking\Carbon_Rates_Database_R&D\app_001_epd\epd_db\1_ec3_global_concrete_rmix_data_to_app.xlsx"
PATH_STEEL = r"Z:\Tier_0\06_Carbon_Rates\Process-based_Carbon_Benchmarking\Carbon_Rates_Database_R&D\app_001_epd\epd_db\1_ec3_global_steel_epd_data_to_app.xlsx" 
PATH_CONCREMIX = r"Z:\Tier_0\06_Carbon_Rates\Process-based_Carbon_Benchmarking\Carbon_Rates_Database_R&D\app_001_epd\epd_db\1_ock_global_concrete_rmix_data_to_app.xlsx" 

def json_safe(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value 

In [16]:
df_concrete = pd.read_excel(PATH_CONCRETE, usecols=["region", "country", "material", "product", "kgCO₂eq", "latitude", "longitude"])
df_concrete = df_concrete.astype({"region": str, "country": str, "material": str, "product": str, "kgCO₂eq": float, "latitude": float, "longitude": float})
df_concrete = df_concrete.rename(columns={
    "region": "Region", 
    "country": "Country", 
    "material": "Material", 
    "product": "ProductName", 
    "kgCO₂eq": "KgCO2eq", 
    "latitude": "Latitude", 
    "longitude": "Longitude" 
})
df_steel = pd.read_excel(PATH_STEEL, usecols=["region", "country", "product", "product name", "kgCO₂eq", "latitude", "longitude"])
df_steel = df_steel.astype({"region": str, "country": str, "product": str, "product name": str, "kgCO₂eq": float, "latitude": float, "longitude": float})
df_steel = df_steel.rename(columns={
    "region": "Region", 
    "country": "Country", 
    "product": "Material", 
    "product name": "ProductName", 
    "kgCO₂eq": "KgCO2eq", 
    "latitude": "Latitude", 
    "longitude": "Longitude" 
})

df = pd.concat([df_concrete, df_steel], ignore_index=True)


In [17]:
df['Latitude'] = df['Latitude'].apply(lambda x: json_safe(x))
df['Longitude'] = df['Longitude'].apply(lambda x: json_safe(x))
df = df.dropna(subset=["Latitude", "Longitude"]) 
res = df.to_dict(orient="records") 
res = [ {key: json_safe(value) for key, value in record.items()} for record in res ] 
df

,Material,Region,Country,ProductName,KgCO2eq,Latitude,Longitude
2,Concrete,Europe,United Kingdom,ReadyMix,389.000000,51.502235,-0.177919
6,Concrete,Europe,United Kingdom,ReadyMix,240.000000,51.502235,-0.177919
12,Concrete,Asia,India,ReadyMix,211.000000,34.170309,77.576974
23,Concrete,Asia,India,ReadyMix,249.000000,34.170309,77.576974
28,Concrete,Oceania,Australia,ReadyMix,158.510000,-33.917536,151.258399
...,...,...,...,...,...,...,...
1380,Cold Formed Decking,Europe,United Kingdom,ComFlor® 80 1.0mm with Colorcoat FD® 170 struc...,3.147854,53.356138,-2.905238
1382,Cold Formed Decking,America,Mexico,Ternium TRD 91.5 Roof Deck,1.787949,23.658512,-102.007710
1383,Cold Formed Decking,America,Mexico,Ternium Losacero 30,1.721000,23.658512,-102.007710
1384,Cold Formed Decking,America,Mexico,Ternium Losacero 15 and 25,1.729000,23.658512,-102.007710


In [19]:
out = {}
for region_idx, (region, region_group) in enumerate(df.groupby("Region")):
    if "children" not in out: 
        out["name"] = "World"
        out["children"] = []
    out["children"].append({
        "name": region,
        "collapsed": False,
        "value": region_group["KgCO2eq"].mean(),
        "children": []
    })
    for country_idx, (country, country_group) in enumerate(region_group.groupby("Country")):
        out["children"][region_idx]["children"].append({
            "name": country,
            "collapsed": False,
            "value": country_group["KgCO2eq"].mean(),
            "children": []
        })
    for material_idx, (material, material_group) in enumerate(region_group.groupby("Material")):
        out["children"][region_idx]["children"][country_idx]["children"].append({
            "name": material,
            "collapsed": True,
            "value": material_group["KgCO2eq"].mean(),
            "children": [
                {
                    "name": row.ProductName, 
                    "value": row.KgCO2eq
                } for row in material_group.itertuples(index=False)
            ]
        })

out

{'name': 'World',
 'children': [{'name': 'America',
   'collapsed': False,
   'value': 13.209733832626426,
   'children': [{'name': 'Brazil',
     'collapsed': False,
     'value': 1.9002500000000002,
     'children': []},
    {'name': 'Canada',
     'collapsed': False,
     'value': 1.2176492180648264,
     'children': []},
    {'name': 'Chile',
     'collapsed': False,
     'value': 1.0248217732311062,
     'children': []},
    {'name': 'Mexico',
     'collapsed': False,
     'value': 100.97655827479552,
     'children': []},
    {'name': 'United States',
     'collapsed': False,
     'value': 7.691993930495138,
     'children': [{'name': 'Coil Steel',
       'collapsed': True,
       'value': 1.894572222222222,
       'children': [{'name': 'Hot-Rolled Coil', 'value': 2.5},
        {'name': 'Galvalume', 'value': 2.6},
        {'name': 'Silhouette XL® - Suspension System', 'value': 2.39},
        {'name': 'Hot Banded Steel Coil - Low & Medium Carbon, High Strength/Low Alloy',
        